In [1]:
import json
import numpy as np
from glob import glob
from tqdm import tqdm
import os

def new_path(f):
    f = f.replace('.mp3', '.alignment')
    f = f.replace('_processed/', '_processed_alignment/')
    return f

def new_path_audioset(f):
    f = f.replace('.mp3', '.audioset_v2')
    f = f.replace('_processed/', '_processed_audioset_v2/')
    return f

In [2]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')

In [206]:
# files = glob('parlimen-24k-chunk_processed/**/*.mp3')
# files = glob('filtered-24k_processed/**/*.mp3')
files = glob('malaysian-podcast_processed/**/*.mp3')
# files.extend(glob('filtered-24k_processed/**/*.mp3'))
# files.extend(glob('malaysian-podcast_processed/**/*.mp3'))

In [207]:
filtered_files = []
for f in tqdm(files):
    f_audioset = new_path_audioset(f)
    if not os.path.exists(f_audioset):
        continue
    f_alignment = new_path(f)
    if not os.path.exists(f_alignment):
        continue
    if os.path.getsize(f_audioset) > 10:
        filtered_files.append((f, f_audioset, f_alignment))
    
len(filtered_files)

100%|███████████████████████████████| 213164/213164 [00:00<00:00, 333277.03it/s]


2621

In [208]:
def get_label(l, return_none = True):
    l_lower = l.lower()
    if 'shouting' in l_lower:
        return 'Shouting'
    if l in ['Bellow', 'Whoop', 'Yell', 'Battle cry', 'Children shouting']:
        return 'Shouting'
    if l in ['Screaming']:
        return 'Screaming'
    if l in ['Whispering']:
        return 'Whispering'
    if 'laugh' in l_lower:
        return 'Laugh'
    if l in ['Snicker', 'Laughter']:
        return 'Laugh'
    if l in ['Giggle']:
        return 'Giggle'
    if 'crying' in l_lower:
        return 'Crying'
    if l in ['Whimper', 'Baby cry, infant cry']:
        return 'Crying'
    if l in ['Sigh']:
        return 'Sigh'
    if l in ['Groan']:
        return 'Groan'
    if l in ['Grunt']:
        return 'Grunt'
    if l in ['Gasp']:
        return 'Gasp'
    if 'Burping' in l:
        return 'Burping'
    if 'Applause' in l:
        return 'Applause'
    if 'Hiccup' in l:
        return 'Hiccup'
    if 'Wail, moan' in l:
        return 'Wail, moan'
    if 'Throat clearing' in l:
        return 'Throat clearing'
    if 'Finger snapping' in l:
        return 'Finger snapping'
    if 'Humming' in l:
        return 'Humming'
    if 'speech' in l_lower:
        return 'Speech'
    if 'monologue' in l_lower:
        return 'Speech'
    if return_none:
        return None
    return l
    
def merge_labels(segments):
    merged = []
    
    for segment in segments:
        if merged and merged[-1]['text'] == segment['text'] and merged[-1]['end'] >= segment['start']:
            merged[-1]['end'] = max(merged[-1]['end'], segment['end'])
        else:
            merged.append(segment)
    
    return merged

def insert_timestamp(timestamps, new_timestamp):
    result = timestamps.copy()
    
    insert_position = 0
    for i, ts in enumerate(timestamps):
        if new_timestamp['start'] < ts['start']:
            insert_position = i
            break
        elif i == len(timestamps) - 1:
            insert_position = i + 1
    
    result.insert(insert_position, new_timestamp)
    
    return result

In [209]:
n = 24
with open(filtered_files[n][1]) as fopen:
    d = json.load(fopen)
    
already_timestamp = set()
filtered = []
for k in range(len(d)):
    t = round(float(d[k]['timestamp']), 2)
    if t in already_timestamp:
        continue
    
    labels = [get_label(l) for no, l in enumerate(d[k]['topk']) if d[k]['scores'][no] >= -4]
    already = set()
    new_labels = []
    for l in labels:
        if l is None:
            continue
        if l not in already:
            new_labels.append(l)
            already.add(l)
    new_new_labels = []
    for l in new_labels:
        if l == 'Speech':
            continue
        else:
            new_new_labels.append(l)
    if len(new_new_labels):
        l = ', '.join(new_new_labels[:2])
        filtered.append({
            'text': f'<|{l}|>',
            'start': t,
            'end': round(float(d[k]['timestamp']) + 0.2, 2),
        })
        already_timestamp.add(t)
        
labels = merge_labels(filtered)

with open(filtered_files[n][2]) as fopen:
    alignment = json.load(fopen)

for l in labels:
    alignment = insert_timestamp(alignment, l)
    
alignment

[{'start': 0.08, 'end': 0.14, 'text': 'Saya', 'score': -10.8515625},
 {'start': 0.22, 'end': 0.42, 'text': 'minta', 'score': -2.3692626953125},
 {'start': 0.52, 'end': 0.68, 'text': 'sebab', 'score': -1.555328369140625},
 {'start': 0.74, 'end': 1.02, 'text': 'diorang,', 'score': -5.086773872375488},
 {'text': '<|Throat clearing, Gasp|>', 'start': 1.0, 'end': 1.4},
 {'start': 1.4, 'end': 1.48, 'text': 'tak', 'score': -0.19019317626953125},
 {'text': '<|Throat clearing|>', 'start': 1.4, 'end': 1.8},
 {'start': 1.54, 'end': 1.7, 'text': 'tahulah', 'score': -14.640426635742188},
 {'start': 1.74, 'end': 1.92, 'text': 'mungkin', 'score': -1.2590599060058594},
 {'start': 1.96, 'end': 2.28, 'text': 'manager', 'score': -6.750034332275391},
 {'start': 2.34, 'end': 2.5, 'text': 'diorang', 'score': -3.434906005859375},
 {'text': '<|Throat clearing|>', 'start': 2.4, 'end': 2.8},
 {'start': 2.54, 'end': 2.62, 'text': 'tak', 'score': -0.46136474609375},
 {'start': 2.66, 'end': 2.84, 'text': 'kasi', '

In [211]:
import IPython.display as ipd
ipd.Audio(filtered_files[n][0])

In [205]:
alignment

[{'start': 0.06,
  'end': 0.68,
  'text': 'Mudah-mudahan,',
  'score': -0.6676164865493774},
 {'start': 1.12,
  'end': 1.68,
  'text': 'berlebih-lebih',
  'score': -0.99090576171875},
 {'start': 1.76,
  'end': 2.2,
  'text': 'keamanan,',
  'score': -0.1683821678161621},
 {'start': 2.7,
  'end': 3.24,
  'text': 'kemakmuran',
  'score': -0.7410173416137695},
 {'start': 3.3, 'end': 3.4, 'text': 'dan', 'score': -0.011850118637084961},
 {'start': 3.5,
  'end': 4.5,
  'text': 'kesejahteraan.',
  'score': -2.527892589569092}]